# Strategy Calibration Notebook  (v2)
Reads historical price and trade CSVs, engineers features, calibrates all trading parameters, and writes a `params.json` ready to paste into `trader_prosperity4.py`.

**What's new in v2:**
- **Fill-rate optimisation**: sweeps quote offsets and finds the one that maximises realised PnL against historical trades, replacing the old "half-spread minus 2" heuristic.
- **Market-making quantity sizer**: replaces the Kelly/volatility formula (which produced qty=1 for stable products) with a sizer that asks how many units you can safely turn over per tick.
- **Bot spread detection**: identifies the dominant bot bid/ask levels so the trader knows where to clamp its quotes.
- Longer ML training (more iterations, more permutation repeats) — no 60s constraint.

**How to use:**
1. Set `PRICE_FILES` and `TRADE_FILES` below.
2. `Kernel → Restart & Run All`.
3. Copy the printed `PARAMS` block from the final cell into `trader_prosperity4.py`.

## 1. Imports

In [ ]:
import json
import warnings
from itertools import product as iterproduct
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")
print("Imports OK")

## 2. Configuration

In [ ]:
# ── Edit these paths ──────────────────────────────────────────────────────
PRICE_FILES = [
    r"prices_round_0_day_-1.csv",
    r"prices_round_0_day_-2.csv",
]
TRADE_FILES = [
    r"trades_round_0_day_-1.csv",
    r"trades_round_0_day_-2.csv",
]
OUTPUT_PARAMS = "params.json"

# ── Calibration settings ───────────────────────────────────────────────────
POSITION_LIMIT_HARD = 80          # IMC Prosperity 4 hard limit per product
POSITION_LIMIT_SOFT = 60          # soft cap used in sizing
LABEL_HORIZON       = 10          # steps ahead for ML label
TRAIN_FRAC          = 0.70

# ML model — more iterations now that we're not time-constrained
MODEL_CONFIG = dict(
    max_iter=400,
    max_depth=5,
    learning_rate=0.04,
    min_samples_leaf=60,
    l2_regularization=1.5,
    random_state=42,
)

# Fill-rate sweep: offsets to test (ticks from fair value)
OFFSET_SWEEP = list(range(1, 10))

print(f"Price files : {PRICE_FILES}")
print(f"Trade files : {TRADE_FILES}")
print(f"Output      : {OUTPUT_PARAMS}")

## 3. Load Data

In [ ]:
def load_data(price_files, trade_files):
    prices = pd.concat([pd.read_csv(f, sep=";") for f in price_files], ignore_index=True)
    trades = pd.concat([pd.read_csv(f, sep=";") for f in trade_files], ignore_index=True)
    prices = prices.sort_values(["product", "day", "timestamp"]).reset_index(drop=True)
    trades = trades.sort_values(["symbol", "timestamp"]).reset_index(drop=True)
    return prices, trades

prices, trades = load_data(PRICE_FILES, TRADE_FILES)
print(f"Prices : {prices.shape[0]:,} rows × {prices.shape[1]} cols")
print(f"Trades : {trades.shape[0]:,} rows × {trades.shape[1]} cols")
print(f"Products: {prices['product'].unique().tolist()}")
print(f"Days    : {sorted(prices['day'].unique().tolist())}")

### Data preview

In [ ]:
prices.head(4)

In [ ]:
trades.head(4)

## 4. Feature Engineering

In [ ]:
def engineer_features(prices, trades):
    df = prices.copy()

    # ── Static order-book features ─────────────────────────────────────────
    df["spread"]          = df["ask_price_1"] - df["bid_price_1"]
    df["spread_pct"]      = df["spread"] / df["mid_price"]
    bid1, ask1            = df["bid_volume_1"], df["ask_volume_1"]
    df["obi_l1"]          = (bid1 - ask1) / (bid1 + ask1)

    bv = df["bid_volume_1"].fillna(0)*3 + df["bid_volume_2"].fillna(0)*2 + df["bid_volume_3"].fillna(0)
    av = df["ask_volume_1"].fillna(0)*3 + df["ask_volume_2"].fillna(0)*2 + df["ask_volume_3"].fillna(0)
    df["obi_weighted"]    = (bv - av) / (bv + av + 1e-9)

    df["total_bid_depth"] = df["bid_volume_1"].fillna(0) + df["bid_volume_2"].fillna(0) + df["bid_volume_3"].fillna(0)
    df["total_ask_depth"] = df["ask_volume_1"].fillna(0) + df["ask_volume_2"].fillna(0) + df["ask_volume_3"].fillna(0)
    df["depth_imbalance"] = (df["total_bid_depth"] - df["total_ask_depth"]) / (df["total_bid_depth"] + df["total_ask_depth"] + 1e-9)

    df["microprice"]      = (df["bid_price_1"]*df["ask_volume_1"] + df["ask_price_1"]*df["bid_volume_1"]) / (df["bid_volume_1"] + df["ask_volume_1"])
    df["microprice_dev"]  = df["microprice"] - df["mid_price"]
    df["bid_gap_12"]      = (df["bid_price_1"] - df["bid_price_2"]).fillna(0)
    df["ask_gap_12"]      = (df["ask_price_2"] - df["ask_price_1"]).fillna(0)

    # ── Rolling features per product ───────────────────────────────────────
    for product, grp in df.groupby("product"):
        idx = grp.index
        for w in [5, 10, 20]:
            df.loc[idx, f"momentum_{w}"]   = grp["mid_price"].pct_change(w)
            df.loc[idx, f"volatility_{w}"] = grp["mid_price"].pct_change().rolling(w).std()
            df.loc[idx, f"obi_mean_{w}"]   = grp["obi_l1"].rolling(w).mean()
        rm = grp["mid_price"].rolling(20).mean()
        rs = grp["mid_price"].rolling(20).std()
        df.loc[idx, "zscore_20"] = (grp["mid_price"] - rm) / (rs + 1e-9)
        df.loc[idx, "ema_8"]     = grp["mid_price"].ewm(span=8,  adjust=False).mean()
        df.loc[idx, "ema_21"]    = grp["mid_price"].ewm(span=21, adjust=False).mean()
        df.loc[idx, "ema_cross"] = df.loc[idx, "ema_8"] - df.loc[idx, "ema_21"]

    # ── Regime dummies ─────────────────────────────────────────────────────
    for product, grp in df.groupby("product"):
        idx     = grp.index
        vol_q75 = grp["volatility_20"].quantile(0.75)
        df.loc[idx, "regime"] = "neutral"
        df.loc[idx[grp["volatility_20"] > vol_q75], "regime"] = "high_vol"
    df = pd.get_dummies(df, columns=["regime"], prefix="regime")

    # ── Interaction features ───────────────────────────────────────────────
    df["obi_x_thin"] = df["obi_l1"] * (1 / (df["total_bid_depth"] + df["total_ask_depth"] + 1e-9))
    df["mom_x_vol"]  = df["momentum_10"] * df["volatility_20"]

    # ── Trade flow (Lee-Ready OFI) ─────────────────────────────────────────
    for product in df["product"].unique():
        sub_p = df[df["product"] == product].copy()
        sub_t = trades[trades["symbol"] == product].copy().sort_values("timestamp")
        if sub_t.empty:
            continue
        merged = pd.merge_asof(sub_t, sub_p[["timestamp", "mid_price"]].sort_values("timestamp"),
                               on="timestamp", direction="backward")
        merged["direction"]  = np.sign(merged["price"] - merged["mid_price"])
        merged["signed_qty"] = merged["direction"] * merged["quantity"]
        for w in [10, 20]:
            window_ts = w * 100
            net_vol = (merged.set_index("timestamp")["signed_qty"]
                       .groupby(merged.set_index("timestamp").index).sum()
                       .reindex(sub_p["timestamp"], method="ffill", fill_value=0)
                       .rolling(window=window_ts, min_periods=1).sum().values)
            tot_vol = (merged.set_index("timestamp")["quantity"]
                       .groupby(merged.set_index("timestamp").index).sum()
                       .reindex(sub_p["timestamp"], method="ffill", fill_value=0)
                       .rolling(window=window_ts, min_periods=1).sum().values)
            df.loc[df["product"] == product, f"ofi_{w}"] = net_vol / (tot_vol + 1e-9)

    # ── Labels ─────────────────────────────────────────────────────────────
    for product, grp in df.groupby("product"):
        idx    = grp.index
        future = grp["mid_price"].shift(-LABEL_HORIZON)
        ret    = (future - grp["mid_price"]) / grp["mid_price"]
        df.loc[idx, "label"] = (ret > 0).astype(float)
        df.loc[idx[-LABEL_HORIZON:], "label"] = np.nan

    return df

df = engineer_features(prices, trades)
print(f"Feature matrix: {df.shape[0]:,} rows × {df.shape[1]} cols")

## 5. Bot Spread Detection
Identifies the dominant resting bid and ask prices posted by bots. These become the bounds for our quote clamping — we want to post *inside* this spread to get queue priority.

In [ ]:
def detect_bot_spread(df, product):
    """
    Find the most common bid1 and ask1 prices (the persistent bot levels).
    Returns (bot_bid, bot_ask) — the dominant resting quotes we must beat.
    """
    sub      = df[df["product"] == product]
    bot_bid  = int(sub["bid_price_1"].mode().iloc[0])
    bot_ask  = int(sub["ask_price_1"].mode().iloc[0])
    bot_spread = bot_ask - bot_bid

    # Frequency analysis
    bid_freq = sub["bid_price_1"].value_counts(normalize=True).head(5)
    ask_freq = sub["ask_price_1"].value_counts(normalize=True).head(5)

    print(f"  {product}:")
    print(f"    Dominant bot bid: {bot_bid}  ({bid_freq.iloc[0]*100:.1f}% of ticks)")
    print(f"    Dominant bot ask: {bot_ask}  ({ask_freq.iloc[0]*100:.1f}% of ticks)")
    print(f"    Bot spread:       {bot_spread} ticks")
    print(f"    Top bid levels:   {bid_freq.index.tolist()}")
    print(f"    Top ask levels:   {ask_freq.index.tolist()}")
    return bot_bid, bot_ask

bot_spreads = {}
for product in sorted(df["product"].unique()):
    bb, ba = detect_bot_spread(df, product)
    bot_spreads[product] = {"bot_bid": bb, "bot_ask": ba, "bot_spread": ba - bb}

### Bot spread visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, product in zip(axes, sorted(df["product"].unique())):
    sub = df[df["product"] == product]
    bid_counts = sub["bid_price_1"].value_counts().head(8).sort_index()
    ask_counts = sub["ask_price_1"].value_counts().head(8).sort_index()
    ax.bar([str(p) for p in bid_counts.index], bid_counts.values, color="#4C72B0", alpha=0.7, label="bid_price_1")
    ax.bar([str(p) for p in ask_counts.index], ask_counts.values, color="#DD8452", alpha=0.7, label="ask_price_1")
    ax.set_title(f"{product} — most common bot quote levels", fontsize=11)
    ax.set_xlabel("Price"); ax.set_ylabel("Tick count")
    ax.tick_params(axis="x", rotation=45)
    ax.legend()
plt.tight_layout()
plt.show()

## 6. Fill-Rate Optimisation
For each candidate quote offset, simulate how many historical market trades would have crossed our passive limit order (giving us queue priority over the bot), and compute the expected PnL at that offset.

This replaces the old "half-spread minus 2" heuristic with something grounded in actual fill data.

In [ ]:
def simulate_fill_rate(df, trades, product, bot_bid, bot_ask, offsets=OFFSET_SWEEP):
    """
    For each offset, simulate passive market-making fills.

    A fill on the BID side happens when a market trade executes at a price
    <= our bid (we post above bot_bid to get queue priority).
    A fill on the ASK side happens when a market trade executes at a price
    >= our ask (we post below bot_ask to get queue priority).

    Returns a DataFrame of results indexed by offset.
    """
    sub_p  = df[df["product"] == product].copy().sort_values("timestamp")
    sub_t  = trades[trades["symbol"] == product].copy().sort_values("timestamp")

    if sub_t.empty:
        return pd.DataFrame()

    # Merge trade prices onto price snapshots for mid-price at trade time
    sub_t = pd.merge_asof(sub_t, sub_p[["timestamp", "mid_price"]],
                           on="timestamp", direction="backward")

    results = []
    for offset in offsets:
        bid_fills = ask_fills = 0
        bid_pnl   = ask_pnl   = 0.0

        for _, row in sub_t.iterrows():
            trade_price = row["price"]
            mid         = row["mid_price"]
            fair        = mid  # use mid as fair value proxy

            our_bid = fair - offset
            our_ask = fair + offset

            # Only quote inside the bot spread
            our_bid = min(our_bid, bot_bid + 1)  # must beat the bot
            our_ask = max(our_ask, bot_ask - 1)

            # Bid fill: incoming sell crosses our bid
            if trade_price <= our_bid:
                bid_fills += 1
                bid_pnl   += (fair - our_bid)   # edge captured

            # Ask fill: incoming buy crosses our ask
            if trade_price >= our_ask:
                ask_fills += 1
                ask_pnl   += (our_ask - fair)   # edge captured

        total_fills = bid_fills + ask_fills
        total_pnl   = bid_pnl + ask_pnl
        pnl_per_fill = total_pnl / total_fills if total_fills > 0 else 0

        results.append({
            "offset":        offset,
            "bid_fills":     bid_fills,
            "ask_fills":     ask_fills,
            "total_fills":   total_fills,
            "bid_pnl":       round(bid_pnl, 2),
            "ask_pnl":       round(ask_pnl, 2),
            "total_pnl":     round(total_pnl, 2),
            "pnl_per_fill":  round(pnl_per_fill, 3),
            "fill_rate_pct": round(total_fills / len(sub_t) * 100, 1),
        })

    return pd.DataFrame(results).set_index("offset")

fill_sweep = {}
for product in sorted(df["product"].unique()):
    bb = bot_spreads[product]["bot_bid"]
    ba = bot_spreads[product]["bot_ask"]
    print(f"\n=== {product} fill-rate sweep (bot: {bb}/{ba}) ===")
    sweep = simulate_fill_rate(df, trades, product, bb, ba)
    fill_sweep[product] = sweep
    print(sweep[["bid_fills","ask_fills","total_fills","total_pnl","pnl_per_fill","fill_rate_pct"]].to_string())

### Fill-rate vs PnL curve — choosing optimal offset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
optimal_offsets = {}

for ax, product in zip(axes, sorted(fill_sweep.keys())):
    sweep = fill_sweep[product]
    if sweep.empty:
        continue

    ax2 = ax.twinx()
    ax.bar(sweep.index, sweep["total_fills"], color="#4C72B0", alpha=0.5, label="Total fills")
    ax2.plot(sweep.index, sweep["total_pnl"], color="#DD8452", lw=2.5, marker="o", label="Total PnL")

    # Best offset = max total_pnl
    best_offset = int(sweep["total_pnl"].idxmax())
    optimal_offsets[product] = best_offset
    ax2.axvline(best_offset, color="#C44E52", lw=2, ls="--", label=f"Best offset={best_offset}")

    ax.set_title(f"{product} — fills & PnL by quote offset", fontsize=11)
    ax.set_xlabel("Quote offset (ticks from fair value)")
    ax.set_ylabel("Fill count", color="#4C72B0")
    ax2.set_ylabel("Simulated PnL", color="#DD8452")

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.tight_layout()
plt.show()

print("\nOptimal offsets found:")
for product, offset in optimal_offsets.items():
    sweep = fill_sweep[product]
    row   = sweep.loc[offset]
    print(f"  {product}: offset={offset}  fills={row['total_fills']}  PnL={row['total_pnl']}  edge/fill={row['pnl_per_fill']}")

## 7. Market-Making Quantity Sizer
The old Kelly-based sizer computed quantity from volatility and produced `qty=1` for stable products like EMERALDS — correct for a directional bet but wrong for a market-maker.

The market-making sizer instead asks: **given the expected fills per tick and our position limit, what quantity lets us turn over inventory most profitably without hitting the hard limit?**

The core formula:
- **Target turnover**: how many units we want to trade per tick on average
- **Safety factor**: keep expected max position well inside the hard limit  
- **Capped** at `POSITION_LIMIT_SOFT // 2` to leave headroom

In [ ]:
def calibrate_mm_quantity(df, trades, product, optimal_offset, bot_bid, bot_ask,
                           position_limit_hard=POSITION_LIMIT_HARD,
                           position_limit_soft=POSITION_LIMIT_SOFT):
    """
    Size market-making quantity based on expected fill rate and inventory risk.
    """
    sub_p  = df[df["product"] == product].copy()
    sub_t  = trades[trades["symbol"] == product].copy()
    n_ticks = len(sub_p)

    # Expected fills per tick at optimal offset
    if not sub_t.empty and optimal_offset in fill_sweep[product].index:
        row            = fill_sweep[product].loc[optimal_offset]
        fills_per_tick = row["total_fills"] / n_ticks
        pnl_per_fill   = row["pnl_per_fill"]
    else:
        fills_per_tick = 0.01
        pnl_per_fill   = optimal_offset * 0.5

    # Inventory risk: with qty Q and fill_rate F, expected position after N ticks
    # is roughly Q * F * N * (1 - 2*balance_rate).
    # We want max position after a burst of one-sided fills to stay inside soft limit.
    # Burst = top 1% of consecutive same-side fills ~ poisson with lambda=fills_per_tick.
    # Conservative: assume 20 consecutive one-sided fills in a row.
    worst_case_consecutive = 20
    max_qty_from_risk = max(1, int(position_limit_soft / worst_case_consecutive))

    # PnL-maximising quantity: more units = more PnL, but capped by risk
    # Additional cap: never more than half the soft limit in a single order
    qty_cap = position_limit_soft // 2
    base_qty = min(qty_cap, max_qty_from_risk)
    base_qty = max(1, base_qty)

    print(f"  {product}:")
    print(f"    Fills per tick:        {fills_per_tick:.4f}")
    print(f"    PnL per fill (edge):   {pnl_per_fill:.2f} ticks")
    print(f"    Max qty from risk:     {max_qty_from_risk}")
    print(f"    Base quantity chosen:  {base_qty}")
    print(f"    Expected PnL / 1000 ticks (single run): "
          f"{fills_per_tick * 1000 * base_qty * pnl_per_fill:.0f}")
    return base_qty, fills_per_tick, pnl_per_fill

mm_quantities = {}
print("=== Market-making quantity calibration ===\n")
for product in sorted(df["product"].unique()):
    bb     = bot_spreads[product]["bot_bid"]
    ba     = bot_spreads[product]["bot_ask"]
    offset = optimal_offsets.get(product, 3)
    qty, fpt, edge = calibrate_mm_quantity(df, trades, product, offset, bb, ba)
    mm_quantities[product] = {"base_quantity": qty, "fills_per_tick": fpt, "edge_per_fill": edge}
    print()

## 8. Microstructure Calibration
Derives the remaining rule-based parameters.

In [ ]:
def calibrate_microstructure(df, product, optimal_offset, base_quantity):
    sub       = df[df["product"] == product].copy().reset_index(drop=True)
    mid       = sub["mid_price"]
    spread    = sub["ask_price_1"] - sub["bid_price_1"]
    ret       = mid.pct_change().dropna()
    zscore    = sub["zscore_20"].dropna()
    obi       = sub["obi_l1"]
    ema_cross = sub["ema_cross"].dropna()
    bb        = bot_spreads[product]["bot_bid"]
    ba        = bot_spreads[product]["bot_ask"]

    micro = {
        "fair_value":        round(float(mid.mean()), 2),
        "fair_value_std":    round(float(mid.std()), 4),
        "typical_spread":    int(spread.median()),
        "spread_std":        round(float(spread.std()), 2),
        "min_spread":        int(spread.min()),
        "max_spread":        int(spread.max()),
        "mid_p01":           round(float(mid.quantile(0.01)), 1),
        "mid_p99":           round(float(mid.quantile(0.99)), 1),
        "avg_bid_depth_l1":  round(float(sub["bid_volume_1"].mean()), 1),
        "avg_ask_depth_l1":  round(float(sub["ask_volume_1"].mean()), 1),
        "return_autocorr_1": round(float(ret.autocorr(lag=1)), 4),
        "return_autocorr_5": round(float(ret.autocorr(lag=5)), 4),
    }

    market_making = {
        "quote_offset":       optimal_offset,            # ← from fill-rate sweep
        "base_quantity":      base_quantity,              # ← from MM sizer
        "bot_bid":            bb,
        "bot_ask":            ba,
        "obi_skew_threshold": round(float(obi.abs().quantile(0.75)), 3),
        "obi_skew_ticks":     1,
        "inventory_lean":     0.4,
        "max_position":       POSITION_LIMIT_SOFT,
        "position_limit_hard": POSITION_LIMIT_HARD,
    }

    mean_reversion = {
        "zscore_entry":   round(float(zscore.abs().quantile(0.90)), 3),
        "zscore_exit":    round(float(zscore.abs().quantile(0.40)), 3),
        "zscore_stop":    round(float(zscore.abs().quantile(0.99)), 3),
        "ema_cross_confirm": round(float(ema_cross.abs().quantile(0.80)), 4),
    }

    # Volatility scalar for momentum adjustment (TOMATOES)
    daily_vol = float(ret.std()) * 100   # in ticks

    return micro, market_making, mean_reversion, daily_vol

all_micro = {}
for product in sorted(df["product"].unique()):
    offset   = optimal_offsets.get(product, 3)
    base_qty = mm_quantities[product]["base_quantity"]
    micro, mm, mr, vol = calibrate_microstructure(df, product, offset, base_qty)
    all_micro[product] = {"micro": micro, "mm": mm, "mr": mr, "vol": vol}
    print(f"\n{'='*52}\n  {product}\n{'='*52}")
    print(f"  Fair value:          {micro['fair_value']}")
    print(f"  Return autocorr 1:   {micro['return_autocorr_1']}")
    print(f"  Quote offset:        {mm['quote_offset']}  (optimised)")
    print(f"  Base quantity:       {mm['base_quantity']}  (MM sizer)")
    print(f"  Bot bid/ask:         {mm['bot_bid']} / {mm['bot_ask']}")
    print(f"  OBI skew threshold:  {mm['obi_skew_threshold']}")
    print(f"  Z-score entry:       {mr['zscore_entry']}")
    print(f"  Z-score stop:        {mr['zscore_stop']}")

### Spread and z-score distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for col, product in enumerate(sorted(df["product"].unique())):
    sub    = df[df["product"] == product]
    spread = sub["ask_price_1"] - sub["bid_price_1"]
    zscore = sub["zscore_20"].dropna()
    micro  = all_micro[product]["micro"]
    mr     = all_micro[product]["mr"]

    ax = axes[0][col]
    ax.hist(spread, bins=30, color="#4C72B0", edgecolor="white", lw=0.4)
    ax.axvline(micro["typical_spread"], color="#DD8452", lw=2, label=f"Median={micro['typical_spread']}")
    ax.set_title(f"{product} — spread distribution", fontsize=11)
    ax.set_xlabel("Spread (ticks)"); ax.set_ylabel("Count"); ax.legend()

    ax = axes[1][col]
    ax.hist(zscore.clip(-5, 5), bins=60, color="#55A868", edgecolor="white", lw=0.4, density=True)
    for thresh, color, label in [
        (mr["zscore_entry"], "#DD8452", f"Entry ±{mr['zscore_entry']:.2f}"),
        (mr["zscore_stop"],  "#C44E52", f"Stop  ±{mr['zscore_stop']:.2f}"),
    ]:
        ax.axvline( thresh, color=color, lw=1.8, ls="--", label=label)
        ax.axvline(-thresh, color=color, lw=1.8, ls="--")
    ax.set_title(f"{product} — z-score distribution", fontsize=11)
    ax.set_xlabel("Z-score"); ax.set_ylabel("Density"); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 9. ML Model Training
Direction classifier per product using walk-forward validation. More iterations than v1 since we're no longer time-constrained.

In [ ]:
EXCL = {"day", "timestamp", "product", "mid_price", "profit_and_loss",
        "mid_price_calc", "microprice", "ema_8", "ema_21", "label"}

def get_feature_cols(sub):
    return [c for c in sub.columns
            if c not in EXCL
            and sub[c].dtype in [np.float64, np.float32, np.int64, np.int32]
            and sub[c].isna().mean() < 0.30]

ml_results = {}

for product in sorted(df["product"].unique()):
    print(f"\n{'='*52}\n  Training: {product}\n{'='*52}")
    sub       = df[df["product"] == product].copy().reset_index(drop=True)
    feat_cols = get_feature_cols(sub)
    sub[feat_cols] = sub[feat_cols].fillna(sub[feat_cols].median())
    sub_ml    = sub.dropna(subset=["label"]).reset_index(drop=True)

    X, y  = sub_ml[feat_cols].values, sub_ml["label"].values
    split = int(len(X) * TRAIN_FRAC)
    X_tr, y_tr = X[:split], y[:split]
    X_te, y_te = X[split:], y[split:]

    print(f"  Features  : {len(feat_cols)}")
    print(f"  Train rows: {split:,}  |  Test rows: {len(X_te):,}")

    model = HistGradientBoostingClassifier(**MODEL_CONFIG)
    model.fit(X_tr, y_tr)

    probs_te = model.predict_proba(X_te)[:, 1]
    auc      = roc_auc_score(y_te, probs_te) if len(np.unique(y_te)) > 1 else None
    acc      = ((probs_te > 0.5).astype(int) == y_te).mean()
    print(f"  Test AUC  : {auc:.4f}")
    print(f"  Test Acc  : {acc:.3f}")

    # More repeats for better importance estimates
    idx_s = np.random.RandomState(42).choice(len(X_te), min(5000, len(X_te)), replace=False)
    pi = permutation_importance(model, X_te[idx_s], y_te[idx_s],
                                n_repeats=20, scoring="roc_auc", random_state=42)
    imp_df = pd.DataFrame({
        "feature":    feat_cols,
        "importance": pi.importances_mean,
        "std":        pi.importances_std,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    top_features = imp_df[imp_df["importance"] > 0].head(15)
    prob_p75     = float(np.percentile(probs_te, 75))
    prob_p90     = float(np.percentile(probs_te, 90))

    ml_results[product] = {
        "model": model, "feat_cols": feat_cols,
        "X_te": X_te, "y_te": y_te, "probs_te": probs_te,
        "auc": auc, "acc": acc, "imp_df": imp_df, "top_features": top_features,
        "prob_p75": prob_p75, "prob_p90": prob_p90,
        "split": split, "n_test": len(X_te),
    }
print("\nTraining complete.")

### Model evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, product in zip(axes, sorted(ml_results.keys())):
    r = ml_results[product]
    RocCurveDisplay.from_predictions(r["y_te"], r["probs_te"], ax=ax,
                                      name=product, color="#4C72B0")
    ax.plot([0,1],[0,1],"--", color="gray", lw=0.8)
    ax.set_title(f"{product}  AUC = {r['auc']:.4f}", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, product in zip(axes, sorted(ml_results.keys())):
    r     = ml_results[product]
    top15 = r["imp_df"].head(15)
    colors = ["#4C72B0" if v > 0 else "#DD8452" for v in top15["importance"]]
    ax.barh(top15["feature"][::-1], top15["importance"][::-1],
            xerr=top15["std"][::-1], color=colors[::-1],
            error_kw={"elinewidth": 0.8, "capsize": 3})
    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"{product} — permutation importance (AUC)", fontsize=12)
    ax.set_xlabel("Mean AUC decrease when feature is shuffled")
plt.tight_layout()
plt.show()

## 10. Assemble & Output `PARAMS`
Assembles the calibrated parameters and prints the `PARAMS` dict exactly as it should appear in `trader_prosperity4.py`. Just copy and paste.

In [ ]:
all_params = {}

for product in sorted(df["product"].unique()):
    micro = all_micro[product]["micro"]
    mm    = all_micro[product]["mm"]
    mr    = all_micro[product]["mr"]
    r     = ml_results[product]

    top_feat_list = [
        {"feature": row["feature"], "importance": round(row["importance"], 5)}
        for _, row in r["top_features"].iterrows()
    ]

    strategy = "market_making"
    use_ml   = bool(r["auc"] and r["auc"] > 0.55)
    vol_to_spread = micro["fair_value_std"] / max(micro["typical_spread"], 1)
    if vol_to_spread > 2.0:
        strategy = "hybrid"

    all_params[product] = {
        # ── copy-paste into trader_prosperity4.py PARAMS dict ────────────
        "product":              product,
        "recommended_strategy": strategy,
        "use_ml_signal":        use_ml,
        "fair_value":           micro["fair_value"],
        "fair_value_std":       micro["fair_value_std"],
        "return_autocorr_1":    micro["return_autocorr_1"],   # informational
        "mid_p01":              micro["mid_p01"],              # informational
        "mid_p99":              micro["mid_p99"],              # informational
        "quote_offset":         mm["quote_offset"],
        "base_quantity":        mm["base_quantity"],
        "bot_bid":              mm["bot_bid"],
        "bot_ask":              mm["bot_ask"],
        "obi_skew_threshold":   mm["obi_skew_threshold"],
        "obi_skew_ticks":       mm["obi_skew_ticks"],
        "inventory_lean":       mm["inventory_lean"],
        "max_position":         mm["max_position"],
        "position_limit_hard":  mm["position_limit_hard"],
        "zscore_entry":         mr["zscore_entry"],
        "zscore_exit":          mr["zscore_exit"],
        "zscore_stop":          mr["zscore_stop"],
        "ema_cross_confirm":    mr["ema_cross_confirm"],
        "ml_auc":               round(r["auc"], 4) if r["auc"] else None,
        "signal_threshold_base":   round(r["prob_p75"], 3),
        "signal_threshold_strong": round(r["prob_p90"], 3),
        "top_features":         top_feat_list,
    }

# ── Save params.json ───────────────────────────────────────────────────────
from pathlib import Path
out_path = Path(OUTPUT_PARAMS)
with open(out_path, "w") as f:
    json.dump(all_params, f, indent=4)
print(f"Saved → {out_path.resolve()}")

# ── Print the exact PARAMS block to paste into trader_prosperity4.py ───────
print()
print("=" * 68)
print("COPY EVERYTHING BELOW INTO trader_prosperity4.py")
print("Replace the existing PARAMS = { ... } block with this:")
print("=" * 68)
print()
# Convert to valid Python (json.dumps outputs true/false/null which are invalid Python)
def to_python_literal(obj, indent=0):
    pad  = "    " * indent
    pad1 = "    " * (indent + 1)
    if isinstance(obj, dict):
        if not obj: return "{}"
        items = [f"{pad1}{repr(k)}: {to_python_literal(v, indent+1)}" for k, v in obj.items()]
        return "{
" + ",
".join(items) + f"
{pad}}}"
    elif isinstance(obj, list):
        if not obj: return "[]"
        items = [f"{pad1}{to_python_literal(v, indent+1)}" for v in obj]
        return "[
" + ",
".join(items) + f"
{pad}]"
    else:
        return repr(obj)   # repr() gives True/False/None — valid Python

print("PARAMS: Dict[str, dict] = " + to_python_literal(all_params))
print()
print("=" * 68)
print("DONE. Paste the block above into trader_prosperity4.py and submit.")

## Done ✓

**Next step:** Copy the `PARAMS` dict printed above and paste it into `trader_prosperity4.py`, replacing the existing `PARAMS` block.

Re-run this notebook each time new competition day data arrives.